In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [ ]:
query = """

WITH legs AS (
    -- entity is buyer = receiver of currency1 (ESMA REFIT GL §435: leg-1 TAKE), so long currency1
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'long' ELSE 'short' END AS direction
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND buyer_nfc = 1
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
      AND notional_eur <= 1e11
    UNION ALL
    -- entity is seller = payer of currency1 (leg-1 MAKE), so short currency1
    SELECT reference_period, CAST(notional_eur AS DOUBLE) AS notional_eur,
           CASE WHEN currency1 <> 'EUR' THEN 'short' ELSE 'long' END AS direction
    FROM lab_prj_emir_ecb.hermesf_fx
    WHERE contract_type = 'FORW' AND seller_nfc = 1
      AND (currency1 = 'EUR' OR currency2 = 'EUR')
      AND notional_eur <= 1e11
)
SELECT
    reference_period,
    SUM(CASE WHEN direction = 'long'  THEN notional_eur ELSE 0 END) AS long_eur,
    SUM(CASE WHEN direction = 'short' THEN notional_eur ELSE 0 END) AS short_eur
FROM legs
GROUP BY reference_period
ORDER BY reference_period

"""

ts = pd.read_sql_query(query, cnxn)

In [ ]:
ts['reference_period'] = pd.to_datetime(ts['reference_period'])
ts = ts.set_index('reference_period').sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(ts.index, ts['long_eur'] / 1e9, label='Long FX')
ax.plot(ts.index, ts['short_eur'] / 1e9, label='Short FX')
ax.set_xlabel('Reference period')
ax.set_ylabel('Outstanding notional (EUR bn)')
ax.set_title('Outstanding forward positions of HermesF entities')
ax.legend()
plt.show()